In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:
df = pd.read_csv("personalised_dataset.csv")

In [3]:
df.columns

Index(['Patient_ID', 'Age', 'Gender', 'BMI', 'Smoking_Status',
       'Alcohol_Consumption', 'Physical_Activity_Level', 'Diet_Type',
       'Blood_Pressure', 'Cholesterol', 'Glucose_Level', 'HbA1c',
       'Heart_Disease_Risk', 'Diabetes_Risk', 'Health_Risk',
       'Predicted_Insurance_Cost', 'Diet_Recommendation',
       'Exercise_Recommendation', 'PRS_Cardiometabolic', 'PRS_Type2Diabetes',
       'APOE_e4_Carrier', 'BRCA_Pathogenic_Variant', 'Family_History_CVD',
       'Family_History_T2D', 'Stress_Level', 'Depression_Score',
       'Anxiety_Score', 'Social_Isolation_Index', 'Sleep_Hours',
       'Sleep_Quality', 'Resting_Heart_Rate', 'HRV', 'Systolic_BP',
       'Diastolic_BP', 'LDL', 'HDL', 'Triglycerides', 'CRP', 'eGFR',
       'Waist_Circumference'],
      dtype='str')

In [4]:
# Data Preparation

def assign_diagnosis(row):
    if row["Systolic_BP"] >= 140 or  row["Diastolic_BP"] >= 90:
        return "Hypertension"
    elif row["Glucose_Level"] >= 126 or row["HbA1c"] >= 6.5:
        return "Diabetes"
    elif row["Cholesterol"] >= 240 or row["LDL"] >= 160 or row["CRP"] >= 3:
        return "Heart Disease"
    elif row["BMI"] < 18.5:
        return "Underweight"
    elif row["BMI"] >= 30 or ( row["Gender"] == "Male" and row["Waist_Circumference"] >= 102 ) or ( row["Gender"] == "Female" and row["Waist_Circumference"] >= 88 ):
        return "Obesity"
    else: 
        return "Healthy"


df["Diagnosis"] = df.apply(assign_diagnosis, axis=1)


med_map = {
    "Hypertension": "ACE inhibitors / Beta-blockers",
    "Diabetes": "Metformin / Insulin",
    "Heart Disease": "Statins / Aspirin",
    "Chronic Kidney Disease": "ACE inhibitors / Dialysis prep",
    "Obesity": "Orlistat / Lifestyle modification",
    "Underweight": "Nutrient-rich diet / Supplements",
    "Healthy": "Fit n Fine! Follow your regular activities and diet"
}

df["Medication"] = df['Diagnosis'].map(med_map)


In [5]:
# Filling Missing values

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="most_frequent")
df["Alcohol_Consumption"] = imputer.fit_transform(df[["Alcohol_Consumption"]]).ravel()


In [6]:
# Ordinal Encoding of features

cols = ["Alcohol_Consumption" , "Heart_Disease_Risk" , "Diabetes_Risk" , "Health_Risk" ,
          "Sleep_Quality" , "Physical_Activity_Level" , "Gender" , "Smoking_Status" , "Diet_Type" ]
order = [
    ["Low", "Moderate", "High"],  # Alcohol_Consumption
    ["Low", "Moderate", "High"],  # Heart_Disease_Risk
    ["Low", "Moderate", "High"],  # Diabetes_Risk
    ["Low", "Moderate", "High"],  # Health_Risk
    ["Poor", "Fair", "Good", "Excellent"],  # Sleep_Quality
    ["Sedentary", "Lightly Active", "Moderately Active", "Highly Active"], # Physical_Activity_Level
    ["Male", "Female"], # Gender
    ["Current smoker", "Former smoker", "Non-smoker"], #Smoking_Status
    ["Vegetarian", "Keto", "Vegan", "Balanced", "High Protein", "Mediterranean"] # Diet_Type
] 

o_enc = OrdinalEncoder(categories=order)
df[cols] = o_enc.fit_transform(df[cols])

In [7]:
# Dropping unnecessary features
df = df.drop(["Patient_ID", "Blood_Pressure"], axis=1)

In [8]:
# Feature and Target selection
X = df.drop(["Cholesterol", "Heart_Disease_Risk", "Diabetes_Risk", "Health_Risk",
        "Predicted_Insurance_Cost", "Diet_Recommendation","Exercise_Recommendation", "Diagnosis", "Medication"] , axis=1)
y_rec = df[["Diet_Recommendation","Exercise_Recommendation", "Diagnosis", "Medication"]]
y_risk = df[["Heart_Disease_Risk", "Diabetes_Risk", "Health_Risk"]]

In [9]:
# Train-Test Split
X_train, X_test, y_rec_train, y_rec_test = train_test_split(
    X, y_rec['Diagnosis'], random_state=42
)

# Training model for predicting diagnosis
gbc = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Train the model
gbc.fit(X_train, y_rec_train)
print(f"{gbc} model trained sucessfully")

# Test the model
predictions = gbc.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_rec_test, predictions)
print(f" Accuracy: {accuracy}")

# Calculate confusion matrix
cm = confusion_matrix(y_rec_test, predictions)
print(f" Confusion Matrix:")
print(np.array2string(cm, separator=', '))


GradientBoostingClassifier(random_state=42) model trained sucessfully
 Accuracy: 0.988
 Confusion Matrix:
[[  6,   0,   0,   0,   0,   0],
 [  0, 282,   2,   0,   0,   0],
 [  0,   3,  83,   0,   0,   0],
 [  0,   0,   0,  20,   0,   0],
 [  0,   0,   0,   0,  82,   0],
 [  0,   0,   1,   0,   0,  21]]


In [10]:
# Training model for predicting risks

heart_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1)
diabetes_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1)
overall_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1)

X_train, X_test, y_train, y_test = train_test_split(X, y_risk["Heart_Disease_Risk"], test_size=0.3, random_state=42)
heart_model.fit(X_train, y_risk["Heart_Disease_Risk"].loc[X_train.index])


X_train, X_test, y_train, y_test = train_test_split(X, y_risk["Diabetes_Risk"], test_size=0.3, random_state=42)
diabetes_model.fit(X_train, y_risk["Diabetes_Risk"].loc[X_train.index])


X_train, X_test, y_train, y_test = train_test_split(X, y_risk["Health_Risk"], test_size=0.3, random_state=42)
overall_model.fit(X_train, y_risk["Health_Risk"].loc[X_train.index])


,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft 

In [11]:
# Recommendation

class HealthRecommender:
    def __init__(self, df, gbc, risk_models, preprocessor=None):
        """
        df: DataFrame containing Diagnosis, Medication, Diet, Exercise columns
        gbc: trained diagnosis classifier
        risk_models: dict of trained risk regressors { "Heart": model, "Diabetes": model, ... }
        preprocessor: optional sklearn pipeline for feature preprocessing
        """
        self.df = df.copy()
        self.df["Diagnosis"] = self.df["Diagnosis"].astype(str).str.strip().str.lower()


        self.gbc = gbc
        self.risk_models = risk_models
        self.preprocessor = preprocessor

    def recommend(self, patient_input):
        # Convert input to DataFrame
        X = pd.DataFrame([patient_input])

        # Apply preprocessing if available
        if self.preprocessor:
            X_transformed = self.preprocessor.transform(X)
        else:
            # Align input with expected features for diagnosis model
            X_transformed = X.reindex(columns=self.gbc.feature_names_in_, fill_value=0)

        # Predict diagnosis
        diagnosis = str(self.gbc.predict(X_transformed)[0]).strip().lower()


        
        # Predict risks
        risks = {}
        for name, model in self.risk_models.items():
            X_aligned = X.reindex(columns=model.feature_names_in_, fill_value=0)
            risks[name] = model.predict(X_aligned)[0]

        # Clean diagnosis column once to avoid whitespace mismatch
        self.df["Diagnosis"] = self.df["Diagnosis"].astype(str).str.strip()


        # Lookup recommendations from dataset
        subset = self.df[self.df["Diagnosis"] == diagnosis]
        if not subset.empty:
            recs = subset.iloc[0][["Medication", "Diet_Recommendation", "Exercise_Recommendation"]].fillna("Consult physician").to_dict()
        else:
            recs = {
                "Medication": "Consult Physician",
                "Diet_Recommendation": "Balanced Diet",
                "Exercise_Recommendation": "Moderate Activity"
                }



        return {
            "Diagnosis": diagnosis,
            "Medication": recs["Medication"],
            "Diet": recs["Diet_Recommendation"],
            "Exercise": recs["Exercise_Recommendation"],
            "Risks": risks
        }


In [12]:

risk_models = {
    "Heart": heart_model,
    "Diabetes": diabetes_model,
    "Overall": overall_model
}

# Instantiate recommender
recommender = HealthRecommender(
    df=df, 
    gbc=gbc, 
    risk_models={
        "Heart": heart_model,
        "Diabetes": diabetes_model,
        "Overall": overall_model
    },
)

# Patient input
patient = {"Age": 45, "BMI": 29, "Glucose_Level": 160}

# Get recommendations
output = recommender.recommend(patient)
print(output)


{'Diagnosis': 'diabetes', 'Medication': 'Metformin / Insulin', 'Diet': 'Low-glycemic, high-fiber; monitor carbs.', 'Exercise': '150+ min moderate cardio + 2x strength/wk; start gradual.', 'Risks': {'Heart': 0.31998843660801474, 'Diabetes': 0.31175439586660786, 'Overall': 0.5227477432935171}}


In [13]:
# Patient input for testing
test_patient = {
    "Age": 35,
    "BMI": 19.3,
    "Glucose_Level": 120,
    "Systolic_BP": 130,
    "Diastolic_BP": 100,
    "eGFR": 150,
    "Smoking_Status":2,
    "LDL": 100,
    "HDL":150
}

# Run through recommender
result = recommender.recommend(test_patient)

print("=== Test Case Result ===")
print(result)


=== Test Case Result ===
{'Diagnosis': 'hypertension', 'Medication': 'ACE inhibitors / Beta-blockers', 'Diet': 'Mediterranean, reduce sat fat & sodium.', 'Exercise': '150+ min moderate cardio + 2x strength/wk; start gradual.', 'Risks': {'Heart': -0.1711141163999559, 'Diabetes': 0.960426578678801, 'Overall': 1.0656996578945606}}
